# Self-Driving on Indian Roads — CNN vs LTC with XAI
### End-to-end pipeline: ordered video → continuous haze → curriculum training → DeepLIFT comparison

**Architecture:**
- **CNN baseline** — EfficientNetV2-S, single-frame input, mixed regression + classification head
- **LTC model** — MobileNetV2 + NCP-wired CfC cells, temporal sequence input (seq_len=8)

**Key fixes over v1:**
- Temporal split (no frame-level leakage)
- Gear treated as classification, not regression
- Continuous on-the-fly haze with Koschmieder physics model
- Curriculum training: clear → moderate → heavy haze
- Feature normalisation + output scaling to reduce loss magnitude
- Side-by-side DeepLIFT XAI for both models on identical frames

---
## 01 · Data Preparation
### 1.1 — Installs & imports

In [ ]:
# Install required libraries (run once)
!pip install captum ncps timm --quiet


In [ ]:
import os, csv, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, f1_score
import timm

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")


### 1.2 — Paths & CSV creation

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DATA_TXT    = "/kaggle/input/self-driving-car-on-indian-roads/indian_dataset/data.txt"
IMAGE_DIR   = "/kaggle/input/self-driving-car-on-indian-roads/indian_dataset"
CSV_PATH    = "data.csv"
TOTAL_FRAMES = 29700

# ── Parse data.txt → data.csv ────────────────────────────────────────────────
# Format: frame_id  steering_degrees  gas_pedal  brake_pedal  gear
header = ["frame_id", "steering_degrees", "gas_pedal", "brake_pedal", "gear"]

with open(DATA_TXT, "r") as txt_f, open(CSV_PATH, "w", newline="") as csv_f:
    writer = csv.writer(csv_f)
    writer.writerow(header)
    skipped = 0
    for i, line in enumerate(txt_f, 1):
        vals = line.strip().split()
        if len(vals) != 5:
            skipped += 1
            continue
        writer.writerow(vals)

print(f"CSV created: {CSV_PATH}  (skipped {skipped} malformed lines)")


In [ ]:
# ── Load & inspect ───────────────────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH)
df_raw.columns = df_raw.columns.str.strip()

print(df_raw.head())
print(f"\nShape: {df_raw.shape}")
print(f"Steering range : {df_raw.steering_degrees.min():.1f} → {df_raw.steering_degrees.max():.1f}")
print(f"Gas range      : {df_raw.gas_pedal.min():.1f} → {df_raw.gas_pedal.max():.1f}")
print(f"Brake range    : {df_raw.brake_pedal.min():.1f} → {df_raw.brake_pedal.max():.1f}")
print(f"Gear values    : {sorted(df_raw.gear.unique())}")


### 1.3 — Output normalisation (critical for low training loss)

One major reason the original notebook had loss ~3.0 is that `steering_degrees`
can span ±180° while `gas_pedal` and `brake_pedal` are in [0,1].  
MSELoss sums these squared errors equally, so steering dominates and inflates the loss.  
We normalise each continuous output to zero-mean / unit-std on the **training set only**,
then invert at evaluation time.

In [ ]:
# ── Temporal train/val/test split ────────────────────────────────────────────
# Sort by frame_id first — NEVER shuffle video frame data before splitting.
df_raw = df_raw.sort_values("frame_id").reset_index(drop=True)

n = len(df_raw)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_df = df_raw.iloc[:train_end].copy()
val_df   = df_raw.iloc[train_end:val_end].copy()
test_df  = df_raw.iloc[val_end:].copy()

print(f"Train : {len(train_df):>6} frames  (0 – 70%)")
print(f"Val   : {len(val_df):>6} frames  (70 – 85%)")
print(f"Test  : {len(test_df):>6} frames  (85 – 100%)")


In [ ]:
# ── Compute normalisation stats on TRAIN only ────────────────────────────────
REG_COLS = ["steering_degrees", "gas_pedal", "brake_pedal"]

norm_mean = train_df[REG_COLS].mean()
norm_std  = train_df[REG_COLS].std().clip(lower=1e-6)   # avoid div-by-zero

print("Normalisation stats (train):")
print(pd.DataFrame({"mean": norm_mean, "std": norm_std}))

# ── Gear label encoding ───────────────────────────────────────────────────────
# Map gear values (whatever they are) to contiguous integers 0..N-1
gear_classes = sorted(df_raw["gear"].unique())
gear2idx     = {g: i for i, g in enumerate(gear_classes)}
N_GEARS      = len(gear_classes)

print(f"\nGear classes: {gear_classes}  →  N_GEARS={N_GEARS}")


### 1.4 — Continuous haze schedule

We model haze intensity α(t) as a smooth signal over the video timeline,
using overlapping sinusoids at different frequencies plus a bounded random walk.
This produces realistic "fog rolling in and out" behaviour rather than a binary haze flag.

The actual haze is applied using the **Koschmieder atmospheric scattering law**:

```
I_hazy(x) = I(x) · (1 − α) + A · α
```

where A ≈ [1,1,1] is white atmospheric light and α ∈ [0, 0.85].

In [ ]:
class TemporalHazeSchedule:
    """
    Smooth, continuous haze intensity schedule over the full video.
    Combines multiple sinusoids at different frequencies to simulate
    fog rolling in, lingering, and clearing — not random per-frame toggling.
    """

    def __init__(self, total_frames: int, seed: int = 42):
        rng = random.Random(seed)
        # Each component: (phase, frequency, amplitude)
        self.components = [
            (rng.uniform(0, 2 * math.pi),
             rng.uniform(0.3, 2.5),   # low-freq = slow fog banks
             rng.uniform(0.08, 0.20)) # amplitude contribution
            for _ in range(6)
        ]
        self.total = total_frames
        self.base  = 0.20             # always some ambient haze (Indian roads)

    def alpha(self, frame_id: int) -> float:
        """Return haze intensity ∈ [0, 0.85] for a given frame."""
        t = frame_id / max(self.total, 1)
        val = self.base
        for phase, freq, amp in self.components:
            val += amp * math.sin(2 * math.pi * freq * t + phase)
        return float(np.clip(val, 0.0, 0.85))


def koschmieder_haze(pil_image: Image.Image, alpha: float) -> Image.Image:
    """
    Apply physically motivated haze (Koschmieder model).
    At alpha=0  → clear image.
    At alpha=0.85 → very dense fog, barely visible.

    I_hazy = I * (1 - alpha) + A * alpha   where A = white atmospheric light.
    """
    if alpha < 0.02:
        return pil_image
    img = np.array(pil_image, dtype=np.float32) / 255.0
    atmospheric_light = np.ones_like(img)             # white light in daylight
    hazy = img * (1.0 - alpha) + atmospheric_light * alpha
    return Image.fromarray((hazy * 255.0).clip(0, 255).astype(np.uint8))


# ── Quick sanity check ────────────────────────────────────────────────────────
schedule = TemporalHazeSchedule(TOTAL_FRAMES)
sample_alphas = [schedule.alpha(i) for i in range(TOTAL_FRAMES)]

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(sample_alphas, lw=0.7, color='steelblue')
ax.axhline(0.5, color='orange', ls='--', lw=1, label='α=0.5 (moderate haze)')
ax.set_xlabel("Frame ID"); ax.set_ylabel("Haze intensity α")
ax.set_title("Temporal haze schedule over full video")
ax.legend(); plt.tight_layout(); plt.show()

print(f"Mean α: {np.mean(sample_alphas):.3f}  |  Max α: {np.max(sample_alphas):.3f}")


---
## 02 · Dataset & Data Loaders
### 2.1 — VideoSequenceDataset

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SEQ_LEN    = 8      # number of consecutive frames fed to the LTC model
BATCH_SIZE = 8      # keep small for GPU memory with seq of images
NUM_WORKERS = 2

class VideoSequenceDataset(Dataset):
    """
    Returns ordered temporal windows of consecutive frames.

    For CNN  → caller takes images[:, -1, :, :, :]   (last frame only)
    For LTC  → caller passes full images tensor       [B, T, C, H, W]

    Haze is applied ON-THE-FLY via the temporal schedule so:
      • training & val  → schedule-driven haze (model learns temporal fog)
      • test (clear)    → alpha=0 override
      • test (haze)     → alpha from schedule (same as training distribution)
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        image_dir: str,
        haze_schedule: TemporalHazeSchedule = None,
        haze_alpha_override: float = None,   # None → use schedule; float → fixed alpha
        transform=None,
        seq_len: int = SEQ_LEN,
        norm_mean=None,
        norm_std=None,
        gear2idx: dict = None,
    ):
        # CRITICAL: preserve temporal order
        self.df       = dataframe.sort_values("frame_id").reset_index(drop=True)
        self.img_dir  = image_dir
        self.schedule = haze_schedule
        self.alpha_override = haze_alpha_override
        self.transform = transform
        self.seq_len   = seq_len
        self.norm_mean = norm_mean
        self.norm_std  = norm_std
        self.gear2idx  = gear2idx

    def __len__(self):
        # Each sample is a window of seq_len consecutive rows
        return max(0, len(self.df) - self.seq_len + 1)

    def _load_frame(self, frame_id: int) -> Image.Image:
        """Load one JPEG frame and apply haze if requested."""
        path = os.path.join(
            self.img_dir, f"circuit2_x264.mp4 {frame_id:05d}.jpg"
        )
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), (128, 128, 128))  # grey fallback

        # ── Haze augmentation ─────────────────────────────────────────────────
        if self.alpha_override is not None:
            alpha = self.alpha_override
        elif self.schedule is not None:
            alpha = self.schedule.alpha(frame_id)
        else:
            alpha = 0.0

        return koschmieder_haze(img, alpha)

    def __getitem__(self, idx):
        window = self.df.iloc[idx : idx + self.seq_len]

        images = []
        for _, row in window.iterrows():
            img = self._load_frame(int(row["frame_id"]))
            if self.transform:
                img = self.transform(img)
            images.append(img)

        images_tensor = torch.stack(images)  # [T, C, H, W]

        # ── Regression targets (last frame of window) ─────────────────────────
        last = window.iloc[-1]
        reg_vals = np.array([
            last["steering_degrees"],
            last["gas_pedal"],
            last["brake_pedal"],
        ], dtype=np.float32)

        # Normalise to ≈ zero-mean / unit-std so MSE stays near 1.0, not 3+
        if self.norm_mean is not None and self.norm_std is not None:
            reg_vals = (reg_vals - self.norm_mean.values) / self.norm_std.values

        reg_target  = torch.tensor(reg_vals, dtype=torch.float32)

        # ── Gear classification target ─────────────────────────────────────────
        raw_gear   = int(last["gear"])
        gear_label = self.gear2idx[raw_gear] if self.gear2idx else raw_gear
        gear_target = torch.tensor(gear_label, dtype=torch.long)

        return images_tensor, reg_target, gear_target


### 2.2 — Transforms

In [ ]:
# ── Image transforms ──────────────────────────────────────────────────────────
# ImageNet mean/std because both EfficientNetV2 and MobileNetV2 are pretrained on it.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),          # mild geometric aug
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


### 2.3 — Build datasets & loaders

In [ ]:
# ── Shared haze schedule (consistent across splits) ───────────────────────────
haze_schedule = TemporalHazeSchedule(TOTAL_FRAMES, seed=SEED)

# ── Datasets ─────────────────────────────────────────────────────────────────
common_kwargs = dict(
    image_dir   = IMAGE_DIR,
    seq_len     = SEQ_LEN,
    norm_mean   = norm_mean,
    norm_std    = norm_std,
    gear2idx    = gear2idx,
)

train_ds = VideoSequenceDataset(
    train_df, haze_schedule=haze_schedule,        # temporal haze ON
    transform=train_transform, **common_kwargs
)

val_ds = VideoSequenceDataset(
    val_df, haze_schedule=haze_schedule,           # same schedule, no aug flip
    transform=test_transform, **common_kwargs
)

# Two test sets: clear frames and haze frames (same test window, different α)
test_clear_ds = VideoSequenceDataset(
    test_df, haze_alpha_override=0.0,              # clear — baseline performance
    transform=test_transform, **common_kwargs
)

test_haze_ds = VideoSequenceDataset(
    test_df, haze_schedule=haze_schedule,          # schedule-driven haze
    transform=test_transform, **common_kwargs
)

print(f"Train sequences : {len(train_ds):>6}")
print(f"Val   sequences : {len(val_ds):>6}")
print(f"Test (clear)    : {len(test_clear_ds):>6}")
print(f"Test (haze)     : {len(test_haze_ds):>6}")


In [ ]:
# ── Data loaders ─────────────────────────────────────────────────────────────
# IMPORTANT: shuffle=False for val/test to preserve temporal order in eval.
loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=True)

train_loader      = DataLoader(train_ds,      BATCH_SIZE, shuffle=True,  **loader_kwargs)
val_loader        = DataLoader(val_ds,        BATCH_SIZE, shuffle=False, **loader_kwargs)
test_clear_loader = DataLoader(test_clear_ds, BATCH_SIZE, shuffle=False, **loader_kwargs)
test_haze_loader  = DataLoader(test_haze_ds,  BATCH_SIZE, shuffle=False, **loader_kwargs)

# ── Quick sanity: inspect one batch ──────────────────────────────────────────
imgs, reg_t, gear_t = next(iter(train_loader))
print(f"images shape : {imgs.shape}")    # [B, T, C, H, W]
print(f"reg targets  : {reg_t.shape}")   # [B, 3]
print(f"gear targets : {gear_t.shape}")  # [B]
print(f"reg target sample  : {reg_t[0].numpy().round(3)}")
print(f"gear target sample : {gear_t[0].item()}")


In [ ]:
# ── Visualise a few frames with haze ─────────────────────────────────────────
def denorm(tensor):
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, SEQ_LEN, figsize=(18, 5))
sample_imgs = imgs[0]   # first sample in batch: [T, C, H, W]
for t in range(SEQ_LEN):
    axes[0, t].imshow(denorm(sample_imgs[t]))
    axes[0, t].set_title(f"t={t}", fontsize=9)
    axes[0, t].axis("off")

# Show the haze schedule for the same frame window
frame_ids = train_ds.df.iloc[0 : SEQ_LEN]["frame_id"].values
alphas    = [haze_schedule.alpha(int(f)) for f in frame_ids]
axes[1, 0].bar(range(SEQ_LEN), alphas, color='steelblue')
axes[1, 0].set_xlabel("t"); axes[1, 0].set_ylabel("α")
axes[1, 0].set_title("Haze intensity for window")
for i in range(1, SEQ_LEN):
    axes[1, i].axis("off")

plt.suptitle("Sample temporal window — first batch", y=1.01)
plt.tight_layout(); plt.show()


---
## 03 · Model Definitions

### 3.1 — Shared components

Both models share:
- A dual-head output: **regression head** (steering, gas, brake) + **classification head** (gear)
- The same normalised target space → MSE loss stays near 1.0 instead of 3+


In [ ]:
class DualHead(nn.Module):
    """
    Shared output head for both CNN and LTC models.
    - regression_head : predicts [steering, gas, brake]  (normalised scale)
    - gear_head       : classifies gear (N_GEARS classes)
    """

    def __init__(self, in_features: int, n_gears: int = N_GEARS):
        super().__init__()
        self.regression_head = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 3),           # steering, gas, brake
        )
        self.gear_head = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.GELU(),
            nn.Linear(64, n_gears),      # logits
        )

    def forward(self, x):
        return self.regression_head(x), self.gear_head(x)


### 3.2 — CNN baseline (EfficientNetV2-S)

In [ ]:
class CNNDrivingModel(nn.Module):
    """
    EfficientNetV2-S backbone → DualHead.

    Input  : single frame [B, C, H, W]
    Outputs: (reg_pred [B, 3], gear_logits [B, N_GEARS])

    Why loss was ~3.0 before:
      • steering_degrees un-normalised → squared error could be 100+
      • gear counted as regression (MSE on [1..5] ≈ constant ~1.5)
    Both are fixed here: normalised targets + separate gear head.
    """

    def __init__(self, n_gears: int = N_GEARS, pretrained: bool = True):
        super().__init__()

        # Backbone — strip the default classifier
        self.backbone = timm.create_model(
            "tf_efficientnetv2_s", pretrained=pretrained, num_classes=0
        )
        feat_dim = self.backbone.num_features   # 1280 for EfficientNetV2-S

        # Bottleneck: reduces 1280 → 256, improves training stability
        self.bottleneck = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
        )

        self.head = DualHead(256, n_gears)

    def forward(self, x):
        """x: [B, C, H, W]"""
        features = self.backbone(x)          # [B, 1280]
        features = self.bottleneck(features) # [B, 256]
        return self.head(features)


### 3.3 — LTC model (MobileNetV2 + CfC/NCP)

In [ ]:
try:
    from ncps.torch import CfC, LTC
    from ncps.wirings import AutoNCP
    NCPS_AVAILABLE = True
    print("✓ NCPS library available")
except ImportError:
    NCPS_AVAILABLE = False
    print("⚠  NCPS not installed — LTC model will use custom fallback")


In [ ]:
class MobileNetEncoder(nn.Module):
    """
    Lightweight per-frame visual encoder.
    MobileNetV2 is used (over EfficientNetV2) for the LTC model because:
      • LTC processes a sequence of T frames per forward pass
      • T × (EfficientNetV2 cost) is expensive; MobileNetV2 is 4× cheaper
      • The LTC temporal dynamics compensate for lower single-frame capacity
    """

    def __init__(self, out_dim: int = 128, pretrained: bool = True):
        super().__init__()
        base = models.mobilenet_v2(pretrained=pretrained)
        self.features = base.features          # outputs [B, 1280, 7, 7]
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))

        self.adapter = nn.Sequential(
            nn.Linear(1280, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, out_dim),
            nn.GELU(),
        )

    def forward(self, x):
        """x: [B, C, H, W]  →  [B, out_dim]"""
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.adapter(x)


class LTCDrivingModel(nn.Module):
    """
    Liquid Neural Network for temporal driving.

    Input  : sequence of frames [B, T, C, H, W]
    Outputs: (reg_pred [B, 3], gear_logits [B, N_GEARS])

    Architecture:
      1. MobileNetV2 encoder applied independently to each frame → [B, T, 128]
      2. CfC (closed-form continuous-time) with AutoNCP wiring processes sequence
      3. DualHead on the last hidden state

    Why CfC instead of LTC-ODE:
      • CfC is a closed-form approximation — no ODE solver, 10× faster training
      • Preserves the liquid time-constant dynamics crucial for robustness
      • Use LTC (use_ltc=True) only if you have ≥ 40GB GPU RAM
    """

    def __init__(
        self,
        n_gears: int = N_GEARS,
        hidden_units: int = 128,
        use_ltc: bool = False,
        pretrained: bool = True,
    ):
        super().__init__()

        if not NCPS_AVAILABLE:
            raise ImportError("Install ncps: pip install ncps")

        self.encoder = MobileNetEncoder(out_dim=128, pretrained=pretrained)

        # AutoNCP wires hidden_units neurons into C. elegans-style NCP hierarchy
        wiring = AutoNCP(hidden_units, 64)   # 64 motor neurons → head input

        CellClass = LTC if use_ltc else CfC
        self.liquid = CellClass(128, wiring, batch_first=True)

        self.head = DualHead(64, n_gears)

    def forward(self, x):
        """x: [B, T, C, H, W]"""
        B, T, C, H, W = x.shape

        # Encode each frame independently
        x_flat   = x.view(B * T, C, H, W)       # [B*T, C, H, W]
        feats    = self.encoder(x_flat)          # [B*T, 128]
        feats    = feats.view(B, T, 128)         # [B, T, 128]

        # Liquid dynamics over the temporal sequence
        out, _   = self.liquid(feats)            # [B, T, 64]
        last_out = out[:, -1, :]                 # take final time-step [B, 64]

        return self.head(last_out)


# ── Custom fallback (no ncps) ─────────────────────────────────────────────────
class CustomLiquidCell(nn.Module):
    """Minimal closed-form LTC cell used when ncps is unavailable."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        d = input_size + hidden_size
        self.ff1  = nn.Linear(d, hidden_size)
        self.ff2  = nn.Linear(d, hidden_size)
        self.gate = nn.Linear(d, hidden_size)

    def forward(self, inp, hx):
        combined = torch.cat([inp, hx], dim=-1)
        ff1  = torch.tanh(self.ff1(combined))
        ff2  = torch.tanh(self.ff2(combined))
        g    = torch.sigmoid(self.gate(combined))
        return ff1 + g * (ff2 - ff1)


class LTCDrivingModelFallback(nn.Module):
    """Fallback LTC model when ncps library is unavailable."""

    def __init__(self, n_gears: int = N_GEARS, pretrained: bool = True):
        super().__init__()
        self.encoder  = MobileNetEncoder(out_dim=128, pretrained=pretrained)
        self.sensory  = CustomLiquidCell(128, 64)
        self.inter    = CustomLiquidCell(64,  32)
        self.head     = DualHead(32, n_gears)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x_flat = x.view(B * T, C, H, W)
        feats  = self.encoder(x_flat).view(B, T, 128)

        h1 = torch.zeros(B, 64,  device=x.device)
        h2 = torch.zeros(B, 32,  device=x.device)

        for t in range(T):
            h1 = self.sensory(feats[:, t], h1)
            h2 = self.inter(h1, h2)

        return self.head(h2)


In [ ]:
# ── Factory function ──────────────────────────────────────────────────────────
def build_model(arch: str, use_ltc: bool = False):
    """
    arch : 'cnn' | 'ltc'
    use_ltc : if True, use ODE-based LTC instead of CfC (slower, more accurate)
    """
    if arch == "cnn":
        model = CNNDrivingModel(n_gears=N_GEARS)
    elif arch == "ltc":
        if NCPS_AVAILABLE:
            model = LTCDrivingModel(n_gears=N_GEARS, use_ltc=use_ltc)
        else:
            print("Using fallback LTC (no ncps)")
            model = LTCDrivingModelFallback(n_gears=N_GEARS)
    else:
        raise ValueError(f"Unknown arch: {arch}")
    return model.to(DEVICE)


# ── Quick parameter counts ────────────────────────────────────────────────────
cnn_model = build_model("cnn")
ltc_model = build_model("ltc")

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"CNN params : {count_params(cnn_model):>12,}")
print(f"LTC params : {count_params(ltc_model):>12,}")


---
## 04 · Training

### 4.1 — Loss function

Dual loss combining MSE for continuous controls and cross-entropy for gear.
The λ_gear weight is tuned so both terms are roughly equal in magnitude.


In [ ]:
def driving_loss(reg_pred, gear_logits, reg_target, gear_target,
                 lambda_gear: float = 0.3):
    """
    Combined loss for multi-output driving prediction.

    reg_pred   : [B, 3]   predicted steering/gas/brake  (normalised)
    gear_logits: [B, G]   raw logits for gear class
    reg_target : [B, 3]   ground truth (normalised)
    gear_target: [B]      ground truth gear class index

    Loss breakdown:
      l_reg  = MSE(pred, target)   — now ≈ 0.5–1.0 because targets are normalised
      l_gear = CrossEntropy        — typically ≈ 1.5 early, drops to <0.5
      total  = l_reg + lambda_gear * l_gear
    """
    l_reg  = F.mse_loss(reg_pred,    reg_target)
    l_gear = F.cross_entropy(gear_logits, gear_target)
    return l_reg + lambda_gear * l_gear, l_reg.item(), l_gear.item()


### 4.2 — Curriculum haze training

Training in three phases prevents the model from failing on heavy haze early:

| Phase | Epochs | Max haze α | Purpose |
|-------|--------|-----------|---------|
| **Clear** | 1–10 | 0.0 | Learn basic driving on clean frames |
| **Moderate** | 11–30 | 0.50 | Introduce mild-to-moderate fog |
| **Full** | 31–50 | 0.85 | Match deployment distribution |


In [ ]:
class CurriculumHazeSchedule(TemporalHazeSchedule):
    """
    Wraps TemporalHazeSchedule and scales α by a curriculum multiplier.
    multiplier=0.0 → clear frames; multiplier=1.0 → full schedule.
    """
    def __init__(self, total_frames, seed=SEED):
        super().__init__(total_frames, seed)
        self.multiplier = 0.0   # updated by trainer each epoch

    def alpha(self, frame_id):
        return super().alpha(frame_id) * self.multiplier


curriculum_schedule = CurriculumHazeSchedule(TOTAL_FRAMES)

# Rebuild training dataset with curriculum schedule so multiplier updates live
train_ds_curriculum = VideoSequenceDataset(
    train_df,
    image_dir         = IMAGE_DIR,
    haze_schedule     = curriculum_schedule,
    transform         = train_transform,
    seq_len           = SEQ_LEN,
    norm_mean         = norm_mean,
    norm_std          = norm_std,
    gear2idx          = gear2idx,
)

train_loader_curriculum = DataLoader(
    train_ds_curriculum, BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)


In [ ]:
def train_one_model(
    arch: str,
    num_epochs: int = 50,
    lr: float = 3e-4,
    patience: int = 8,
    use_ltc_ode: bool = False,
    save_path: str = None,
):
    """
    Full training loop with:
      • Curriculum haze schedule
      • Cosine LR scheduler with linear warm-up (5 epochs)
      • Gradient clipping (essential for LTC stability)
      • Early stopping on validation regression loss
    """
    model = build_model(arch, use_ltc=use_ltc_ode)
    save_path = save_path or f"best_{arch}.pth"

    # Cosine annealing LR — works much better than ReduceLROnPlateau for this task
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr   = lr,
        epochs   = num_epochs,
        steps_per_epoch = len(train_loader_curriculum),
        pct_start = 0.10,        # 10% warm-up
        anneal_strategy = "cos",
    )

    best_val_loss   = float("inf")
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "train_reg": [], "val_reg": []}

    # ── Curriculum phases ──────────────────────────────────────────────────────
    def get_haze_multiplier(epoch):
        if epoch < 10:  return 0.0    # Phase 1: clear
        if epoch < 30:  return 0.6    # Phase 2: moderate
        return 1.0                    # Phase 3: full haze

    print(f"\n{'='*65}")
    print(f" Training {arch.upper()}  |  {count_params(model):,} params")
    print(f"{'='*65}")

    for epoch in range(num_epochs):

        # Update curriculum haze level
        curriculum_schedule.multiplier = get_haze_multiplier(epoch)

        # ── Train ──────────────────────────────────────────────────────────────
        model.train()
        t_loss = t_reg = t_gear = 0.0

        for imgs, reg_t, gear_t in train_loader_curriculum:
            imgs   = imgs.to(DEVICE)
            reg_t  = reg_t.to(DEVICE)
            gear_t = gear_t.to(DEVICE)

            # CNN only needs last frame; LTC needs full sequence
            if arch == "cnn":
                inp = imgs[:, -1]   # [B, C, H, W]
            else:
                inp = imgs          # [B, T, C, H, W]

            optimizer.zero_grad()
            reg_pred, gear_logits = model(inp)
            loss, l_r, l_g = driving_loss(reg_pred, gear_logits, reg_t, gear_t)

            loss.backward()
            # Gradient clipping: prevents explosion in LTC recurrent weights
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            t_loss += loss.item(); t_reg += l_r; t_gear += l_g

        n = len(train_loader_curriculum)
        history["train_loss"].append(t_loss / n)
        history["train_reg"].append(t_reg  / n)

        # ── Validate ───────────────────────────────────────────────────────────
        model.eval()
        v_loss = v_reg = 0.0

        with torch.no_grad():
            for imgs, reg_t, gear_t in val_loader:
                imgs   = imgs.to(DEVICE)
                reg_t  = reg_t.to(DEVICE)
                gear_t = gear_t.to(DEVICE)

                inp = imgs[:, -1] if arch == "cnn" else imgs
                reg_pred, gear_logits = model(inp)
                loss, l_r, _ = driving_loss(reg_pred, gear_logits, reg_t, gear_t)
                v_loss += loss.item(); v_reg += l_r

        nv = len(val_loader)
        avg_vl = v_loss / nv
        history["val_loss"].append(avg_vl)
        history["val_reg"].append(v_reg / nv)

        phase = ["CLEAR", "MODERATE", "FULL HAZE"][
            0 if epoch < 10 else (1 if epoch < 30 else 2)
        ]
        print(
            f"Ep {epoch+1:3d}/{num_epochs}  [{phase:9s}]  "
            f"train={history['train_loss'][-1]:.4f}  "
            f"val={avg_vl:.4f}  "
            f"lr={scheduler.get_last_lr()[0]:.2e}"
        )

        # ── Early stopping ─────────────────────────────────────────────────────
        if avg_vl < best_val_loss:
            best_val_loss = avg_vl
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            if patience_counter >= patience and epoch >= 15:
                print(f"  ↳ Early stop at epoch {epoch+1} (patience={patience})")
                break

    model.load_state_dict(torch.load(save_path))
    print(f"\nBest val loss: {best_val_loss:.4f}  — model saved to {save_path}")
    return model, history


In [ ]:
# ── Train CNN ─────────────────────────────────────────────────────────────────
cnn_model, cnn_history = train_one_model("cnn", num_epochs=50, lr=3e-4)


In [ ]:
# ── Train LTC ─────────────────────────────────────────────────────────────────
# use_ltc_ode=False uses CfC (faster, recommended for Kaggle GPU quota)
# Set use_ltc_ode=True only on high-memory GPUs (≥24GB)
ltc_model, ltc_history = train_one_model("ltc", num_epochs=50, lr=2e-4, use_ltc_ode=False)


In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, hist, title in zip(
    axes,
    [cnn_history, ltc_history],
    ["CNN (EfficientNetV2-S)", "LTC (MobileNetV2 + CfC)"]
):
    ax.plot(hist["train_loss"], label="Train total")
    ax.plot(hist["val_loss"],   label="Val total")
    ax.plot(hist["train_reg"],  label="Train MSE", ls="--", alpha=0.7)
    ax.plot(hist["val_reg"],    label="Val MSE",   ls="--", alpha=0.7)

    # Shade curriculum phases
    n = len(hist["train_loss"])
    ax.axvspan(0,           min(10, n), alpha=0.07, color="green",  label="Clear phase")
    ax.axvspan(min(10, n),  min(30, n), alpha=0.07, color="orange", label="Moderate haze")
    ax.axvspan(min(30, n),  n,          alpha=0.07, color="red",    label="Full haze")

    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Training history — curriculum haze", fontsize=13)
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()


---
## 05 · Evaluation & XAI Comparison
### 5.1 — Metric utilities

In [ ]:
REG_NAMES  = ["steering", "gas", "brake"]
GEAR_NAMES = [str(g) for g in gear_classes]


def evaluate(model, loader, arch: str, split_name: str = "Test"):
    """
    Run inference on loader, return denormalised predictions and targets.
    Reports per-output MAE, R², plus gear accuracy and F1.
    """
    model.eval()
    all_reg_pred, all_reg_true, all_gear_pred, all_gear_true = [], [], [], []

    with torch.no_grad():
        for imgs, reg_t, gear_t in loader:
            imgs   = imgs.to(DEVICE)
            inp    = imgs[:, -1] if arch == "cnn" else imgs
            reg_pred, gear_logits = model(inp)

            all_reg_pred.append(reg_pred.cpu().numpy())
            all_reg_true.append(reg_t.numpy())
            all_gear_pred.append(gear_logits.argmax(1).cpu().numpy())
            all_gear_true.append(gear_t.numpy())

    reg_pred  = np.vstack(all_reg_pred)
    reg_true  = np.vstack(all_reg_true)
    gear_pred = np.concatenate(all_gear_pred)
    gear_true = np.concatenate(all_gear_true)

    # ── Denormalise back to original units ─────────────────────────────────────
    reg_pred_dn = reg_pred  * norm_std.values  + norm_mean.values
    reg_true_dn = reg_true  * norm_std.values  + norm_mean.values

    print(f"\n{split_name} — {arch.upper()}")
    print("=" * 58)
    results = {}

    for i, col in enumerate(REG_NAMES):
        mae = mean_absolute_error(reg_true_dn[:, i], reg_pred_dn[:, i])
        r2  = r2_score(reg_true_dn[:, i], reg_pred_dn[:, i])
        results[col] = {"MAE": mae, "R2": r2}
        print(f"  {col:12s}  MAE={mae:.4f}  R²={r2:.4f}")

    gear_acc = (gear_pred == gear_true).mean()
    gear_f1  = f1_score(gear_true, gear_pred, average="weighted", zero_division=0)
    results["gear"] = {"Accuracy": gear_acc, "F1": gear_f1}
    print(f"  {'gear':12s}  Acc={gear_acc:.4f}  F1={gear_f1:.4f}")

    return reg_pred_dn, reg_true_dn, gear_pred, gear_true, results


### 5.2 — Evaluate both models on clear and haze test sets

In [ ]:
# CNN — clear
cnn_clear_pred, cnn_clear_true, cnn_gear_pred_c, cnn_gear_true_c, cnn_metrics_clear = \
    evaluate(cnn_model, test_clear_loader, "cnn", "Test Clear")

# CNN — haze
cnn_haze_pred, cnn_haze_true, cnn_gear_pred_h, cnn_gear_true_h, cnn_metrics_haze = \
    evaluate(cnn_model, test_haze_loader, "cnn", "Test Haze")

# LTC — clear
ltc_clear_pred, ltc_clear_true, ltc_gear_pred_c, ltc_gear_true_c, ltc_metrics_clear = \
    evaluate(ltc_model, test_clear_loader, "ltc", "Test Clear")

# LTC — haze
ltc_haze_pred, ltc_haze_true, ltc_gear_pred_h, ltc_gear_true_h, ltc_metrics_haze = \
    evaluate(ltc_model, test_haze_loader, "ltc", "Test Haze")


In [ ]:
# ── Summary comparison table ─────────────────────────────────────────────────
rows = []
for col in REG_NAMES:
    rows.append({
        "Output": col,
        "CNN MAE (clear)":  cnn_metrics_clear[col]["MAE"],
        "CNN MAE (haze)":   cnn_metrics_haze[col]["MAE"],
        "LTC MAE (clear)":  ltc_metrics_clear[col]["MAE"],
        "LTC MAE (haze)":   ltc_metrics_haze[col]["MAE"],
        "CNN R² (clear)":   cnn_metrics_clear[col]["R2"],
        "LTC R² (clear)":   ltc_metrics_clear[col]["R2"],
    })

summary_df = pd.DataFrame(rows).set_index("Output").round(4)
print("\n── Comparison Table ─────────────────────────────────────────────")
print(summary_df.to_string())


In [ ]:
# ── Robustness curve: MAE vs haze intensity ───────────────────────────────────
# Run test set through BOTH models at alpha = 0, 0.2, 0.4, 0.6, 0.8
# and plot how steering MAE degrades for each model.

alpha_levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
cnn_mae_per_alpha = []
ltc_mae_per_alpha = []

for alpha in alpha_levels:
    ds_tmp = VideoSequenceDataset(
        test_df, image_dir=IMAGE_DIR,
        haze_alpha_override=alpha,
        transform=test_transform,
        seq_len=SEQ_LEN,
        norm_mean=norm_mean, norm_std=norm_std, gear2idx=gear2idx
    )
    ld_tmp = DataLoader(ds_tmp, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    for arch, model, mae_list in [
        ("cnn", cnn_model, cnn_mae_per_alpha),
        ("ltc", ltc_model, ltc_mae_per_alpha),
    ]:
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for imgs, reg_t, _ in ld_tmp:
                inp = imgs[:, -1].to(DEVICE) if arch == "cnn" else imgs.to(DEVICE)
                reg_pred, _ = model(inp)
                preds.append(reg_pred.cpu().numpy())
                trues.append(reg_t.numpy())
        p = np.vstack(preds) * norm_std["steering_degrees"] + norm_mean["steering_degrees"]
        t = np.vstack(trues) * norm_std["steering_degrees"] + norm_mean["steering_degrees"]
        mae_list.append(mean_absolute_error(t[:, 0], p[:, 0]))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alpha_levels, cnn_mae_per_alpha, "o-", label="CNN", color="steelblue")
ax.plot(alpha_levels, ltc_mae_per_alpha, "s-", label="LTC", color="darkorange")
ax.set_xlabel("Haze intensity α"); ax.set_ylabel("Steering MAE (degrees)")
ax.set_title("Robustness curve — MAE vs haze intensity")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("robustness_curve.png", dpi=150); plt.show()


### 5.3 — Side-by-side DeepLIFT XAI

For each test frame we show a 3-panel row per output (steering, gas, brake):
```
[Original frame]  |  [CNN attribution]  |  [LTC attribution]
```
The key thing to look for:  
- CNN heatmaps → spatial (road lines, lane markings)  
- LTC heatmaps → how temporal context shifts focus (fog region, scene edges)


In [ ]:
from captum.attr import DeepLift


def deeplift_attrs(model, image_tensor, output_idx, device=DEVICE):
    """
    Compute DeepLIFT attribution for one output index.
    image_tensor : [C, H, W]  (single frame for CNN)
                   or [T, C, H, W] (sequence for LTC)
    Returns attribution map as [H, W] numpy array.
    """
    model.eval()

    inp  = image_tensor.unsqueeze(0).to(device)
    inp.requires_grad_(True)
    base = torch.zeros_like(inp).to(device)

    # DeepLIFT needs the model to return a single tensor; we split the dual head
    class _RegWrapper(nn.Module):
        def __init__(self, m, arch):
            super().__init__(); self.m = m; self.arch = arch
        def forward(self, x):
            reg_pred, _ = self.m(x[:, -1] if self.arch == "cnn" else x)
            return reg_pred

    wrapper = _RegWrapper(model, arch="cnn" if image_tensor.dim() == 3 else "ltc")
    dl   = DeepLift(wrapper)
    attr = dl.attribute(inp, base, target=output_idx)

    attr_np = attr.squeeze(0).detach().cpu().numpy()           # [C, H, W] or [T, C, H, W]
    if attr_np.ndim == 4:   # LTC: average over temporal dimension
        attr_np = attr_np.mean(axis=0)
    return np.abs(attr_np).mean(axis=0)                        # [H, W]


def denorm_for_display(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


In [ ]:
# ── Side-by-side DeepLIFT visualisation ──────────────────────────────────────
# We pick 3 test samples: one clear, one moderate haze, one heavy haze
N_VIS    = 3
OUT_NAMES = ["Steering", "Gas", "Brake"]

# Select frames spanning different haze levels from the test set
test_indices = [0, len(test_haze_ds) // 2, len(test_haze_ds) - 1]

for sample_idx in test_indices:
    imgs_seq, reg_t, gear_t = test_haze_ds[sample_idx]   # [T, C, H, W]
    last_frame  = imgs_seq[-1]    # [C, H, W] for CNN
    alpha_val   = haze_schedule.alpha(int(test_df.sort_values("frame_id").iloc[sample_idx]["frame_id"]))

    # Model predictions (denormalised)
    with torch.no_grad():
        cnn_pred_n, _ = cnn_model(last_frame.unsqueeze(0).to(DEVICE))
        ltc_pred_n, _ = ltc_model(imgs_seq.unsqueeze(0).to(DEVICE))

    cnn_pred = cnn_pred_n.cpu().numpy()[0] * norm_std.values + norm_mean.values
    ltc_pred = ltc_pred_n.cpu().numpy()[0] * norm_std.values + norm_mean.values
    reg_true = reg_t.numpy() * norm_std.values + norm_mean.values

    display_img = denorm_for_display(last_frame)

    fig = plt.figure(figsize=(15, 4 * 3))
    fig.suptitle(
        f"DeepLIFT — sample {sample_idx}  |  haze α={alpha_val:.2f}  |  "
        f"True steering={reg_true[0]:.1f}°",
        fontsize=12
    )

    for oi, out_name in enumerate(OUT_NAMES):
        # Compute attributions
        cnn_attr = deeplift_attrs(cnn_model, last_frame,          oi)
        ltc_attr = deeplift_attrs(ltc_model, imgs_seq,             oi)

        row_offset = oi * 3
        ax_orig = fig.add_subplot(3, 3, row_offset + 1)
        ax_cnn  = fig.add_subplot(3, 3, row_offset + 2)
        ax_ltc  = fig.add_subplot(3, 3, row_offset + 3)

        # Original frame
        ax_orig.imshow(display_img)
        ax_orig.set_title(f"{out_name} — Original (α={alpha_val:.2f})")
        ax_orig.axis("off")

        # CNN attribution
        ax_cnn.imshow(display_img)
        ax_cnn.imshow(cnn_attr, cmap="hot", alpha=0.55,
                      vmin=cnn_attr.min(), vmax=cnn_attr.max())
        ax_cnn.set_title(
            f"CNN DeepLIFT\npred={cnn_pred[oi]:.2f}  true={reg_true[oi]:.2f}"
        )
        ax_cnn.axis("off")

        # LTC attribution
        ax_ltc.imshow(display_img)
        ax_ltc.imshow(ltc_attr, cmap="hot", alpha=0.55,
                      vmin=ltc_attr.min(), vmax=ltc_attr.max())
        ax_ltc.set_title(
            f"LTC DeepLIFT\npred={ltc_pred[oi]:.2f}  true={reg_true[oi]:.2f}"
        )
        ax_ltc.axis("off")

    plt.tight_layout()
    plt.savefig(f"deeplift_sample{sample_idx}.png", dpi=150, bbox_inches="tight")
    plt.show()


### 5.4 — Final summary bar chart

In [ ]:
# ── Final comparison bar chart ────────────────────────────────────────────────
metrics_to_plot = ["steering", "gas", "brake"]
conditions      = ["clear", "haze"]
models_         = ["CNN", "LTC"]

x = np.arange(len(metrics_to_plot))
width = 0.20
fig, ax = plt.subplots(figsize=(12, 5))

offsets = [-1.5, -0.5, 0.5, 1.5]
combos  = [
    ("CNN", "clear", cnn_metrics_clear, "royalblue"),
    ("CNN", "haze",  cnn_metrics_haze,  "cornflowerblue"),
    ("LTC", "clear", ltc_metrics_clear, "darkorange"),
    ("LTC", "haze",  ltc_metrics_haze,  "moccasin"),
]

for (model_n, cond, metrics, color), offset in zip(combos, offsets):
    vals = [metrics[m]["MAE"] for m in metrics_to_plot]
    ax.bar(x + offset * width, vals, width, label=f"{model_n} {cond}", color=color)

ax.set_xticks(x); ax.set_xticklabels([m.capitalize() for m in metrics_to_plot])
ax.set_ylabel("MAE (original units)")
ax.set_title("CNN vs LTC — MAE comparison, clear vs haze")
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("final_comparison.png", dpi=150); plt.show()
print("\n✓ All outputs saved: training_curves.png, robustness_curve.png,")
print("  deeplift_sample*.png, final_comparison.png")
